In [ ]:
```python
import pandas as pd
import os
# 1. DATASET LOCATION
folder = "datasets"
# 2. DATASETS
files = {
    "Renewable": "Renewable Energy monitoring.xlsx",
    "SCADA": "scada_pipeline.csv",
    "Smart Meter": "smart_meter_data.csv",
    "Maintenance": "asset_maintenance.csv",
    "Billing": "Billing_payments.csv",
    "IIoT": "iiot_smart_grid_dataset.csv",
    "OWID": "owid-energy-data.json",
    "OMS": "OMS_data.csv"
}
# 3. LOAD DATASETS
def load_file(filename):

    path = os.path.join(folder, filename)

    if filename.endswith(".csv"):
        return pd.read_csv(path)

    elif filename.endswith(".xlsx"):
        return pd.read_excel(path, sheet_name="data")

    elif filename.endswith(".json"):
        return pd.read_json(path)
data = {}
for name, file in files.items():
    try:
        data[name] = load_file(file)
        print(f"✓ {name} loaded")
    except Exception as e:
        print(f"✗ Error loading {name}: {e}")
# 4. DEFINE RELATED COLUMNS
mappings = [

    # Renewable ↔ OWID
    ("Renewable", "country",
     "OWID", "country",
     "Same geographic entity"),

    ("Renewable", "year",
     "OWID", "year",
     "Same time dimension"),

    ("Renewable", "iso_code",
     "OWID", "iso_code",
     "Country ISO identifier"),

    # Smart Meter ↔ IIoT
    ("Smart Meter", "Timestamp",
     "IIoT", "Timestamp",
     "Same time dimension"),

    ("Smart Meter", "Electricity_Consumed",
     "IIoT", "Power_Consumption_kWh",
     "Related energy consumption measure"),

    ("Smart Meter", "Temperature",
     "IIoT", "Temperature_C",
     "Related temperature measure"),

    ("Smart Meter", "Humidity",
     "IIoT", "Humidity_%",
     "Related humidity measure"),

    # SCADA ↔ IIoT
    ("SCADA", "timestamp",
     "IIoT", "Timestamp",
     "Related operational time field"),

    ("SCADA", "energy_consumption",
     "IIoT", "Power_Consumption_kWh",
     "Related energy consumption measure"),

    ("SCADA", "temperature",
     "IIoT", "Temperature_C",
     "Related temperature measure"),

    # SCADA ↔ Smart Meter
    ("SCADA", "timestamp",
     "Smart Meter", "Timestamp",
     "Related time field"),

    ("SCADA", "temperature",
     "Smart Meter", "Temperature",
     "Related temperature measure"),

    # OMS ↔ Billing
    ("OMS", "Year",
     "Billing", "Utility.State",
     "Geographic/time relationship only"),

    ("OMS", "Geographic Areas",
     "Billing", "Utility.State",
     "Geographic relationship"),

    ("OMS", "Number of Customers Affected",
     "Billing", "Retail.Total.Customers",
     "Related customer measure"),
    # OMS ↔ Renewable
    ("OMS", "Year",
     "Renewable", "year",
     "Same time dimension"),
    ("OMS", "Geographic Areas",
     "Renewable", "country",
     "Geographic relationship")
]
# 5. MAPPING ANALYSIS
report = []
for d1, c1, d2, c2, relationship in mappings:
    # Check dataset existence
    if d1 not in data or d2 not in data:
        continue
    df1 = data[d1]
    df2 = data[d2]
    if c1 not in df1.columns or c2 not in df2.columns:
        report.append([
            d1,
            c1,
            d2,
            c2,
            relationship,
            "Column Not Found",
            "",
            "",
            "",
            ""
        ])
        continue
    s1 = df1[c1]
    s2 = df2[c2]
    # Basic information
    type1 = str(s1.dtype)
    type2 = str(s2.dtype)
    missing1 = s1.isna().sum()
    missing2 = s2.isna().sum()
    unique1 = s1.nunique()
    unique2 = s2.nunique()
    # Data type compatibility
    if (
        pd.api.types.is_numeric_dtype(s1)
        and pd.api.types.is_numeric_dtype(s2)
    ):
        type_match = "Compatible"

    elif (
        pd.api.types.is_datetime64_any_dtype(s1)
        and pd.api.types.is_datetime64_any_dtype(s2)
    ):
        type_match = "Compatible"

    elif s1.dtype == s2.dtype:
        type_match = "Compatible"

    else:
        type_match = "Different"
    # Common values
    common_values = ""
    values1 = set(s1.dropna().astype(str))
    values2 = set(s2.dropna().astype(str))
    common_count = len(values1 & values2)
    # Store result
    report.append([
        d1,
        c1,
        d2,
        c2,
        relationship,
        type_match,
        type1,
        type2,
        unique1,
        unique2,
        missing1,
        missing2,
        common_count
    ])
# 6. CREATE MAPPING REPORT
mapping_df = pd.DataFrame(
    report,
    columns=[
        "Dataset 1",
        "Column 1",
        "Dataset 2",
        "Column 2",
        "Relationship",
        "Data Type Compatibility",
        "Type 1",
        "Type 2",
        "Unique Values 1",
        "Unique Values 2",
        "Missing Values 1",
        "Missing Values 2",
        "Common Values"
    ]
)
mapping_df.to_excel(
    "Column_Mapping_Report.xlsx",
    index=False
)
# 7. TIMESTAMP MAPPING
print("\n" + "=" * 70)
print("TIMESTAMP MAPPING")
print("=" * 70)


if "Smart Meter" in data and "IIoT" in data:

    smart_time = pd.to_datetime(
        data["Smart Meter"]["Timestamp"],
        errors="coerce"
    )

    iiot_time = pd.to_datetime(
        data["IIoT"]["Timestamp"],
        errors="coerce"
    )

    common = set(
        smart_time.dropna()
    ) & set(
        iiot_time.dropna()
    )

    print("Smart Meter timestamps:",
          smart_time.nunique())

    print("IIoT timestamps:",
          iiot_time.nunique())

    print("Common timestamps:",
          len(common))
# =========================================================
# 8. COUNTRY-YEAR MAPPING
# =========================================================

print("\n" + "=" * 70)
print("COUNTRY-YEAR MAPPING")
print("=" * 70)


if "Renewable" in data and "OWID" in data:

    renewable_keys = set(
        zip(
            data["Renewable"]["country"],
            data["Renewable"]["year"]
        )
    )

    owid_keys = set(
        zip(
            data["OWID"]["country"],
            data["OWID"]["year"]
        )
    )

    common_keys = (
        renewable_keys &
        owid_keys
    )

    print("Renewable country-year records:",
          len(renewable_keys))

    print("OWID country-year records:",
          len(owid_keys))

    print("Common country-year records:",
          len(common_keys))
print("\n" + "=" * 70)
print("COLUMN MAPPING COMPLETED")
print("=" * 70)
print("Created:")
print("Column_Mapping_Report.xlsx")
